# 03 — Feature Engineering

## Goal
Convert the raw time-series data into a machine-learning-ready dataset.

Features:
- calendar features
- sales lags
- leakage-safe rolling statistics
- promotions
- store information
- transactions
- oil price
- holidays

### Important
For rolling features we shift sales by one day first. Therefore today's target is never used to calculate today's feature.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# The notebooks live inside /notebooks.
# Therefore the project root is one folder above the notebook folder.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(f"Raw data folder not found: {RAW_DATA_DIR}")

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DATA_DIR)
print("Processed data folder:", PROCESSED_DATA_DIR)


Project root: d:\Projects\Revenue Demand Forecasting
Raw data folder: d:\Projects\Revenue Demand Forecasting\data\raw
Processed data folder: d:\Projects\Revenue Demand Forecasting\data\processed


In [3]:
train = pd.read_csv(RAW_DATA_DIR / "train.csv", parse_dates=["date"])
stores = pd.read_csv(RAW_DATA_DIR / "stores.csv")
oil = pd.read_csv(RAW_DATA_DIR / "oil.csv", parse_dates=["date"])
holidays = pd.read_csv(RAW_DATA_DIR / "holidays_events.csv", parse_dates=["date"])
transactions = pd.read_csv(RAW_DATA_DIR / "transactions.csv", parse_dates=["date"])

print("Train shape:", train.shape)


Train shape: (3000888, 6)


In [4]:
# Always sort each store + product family chronologically before creating lags.
df = train.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

print("Sorted:", df[["store_nbr", "family", "date"]].head())


Sorted:    store_nbr      family       date
0          1  AUTOMOTIVE 2013-01-01
1          1  AUTOMOTIVE 2013-01-02
2          1  AUTOMOTIVE 2013-01-03
3          1  AUTOMOTIVE 2013-01-04
4          1  AUTOMOTIVE 2013-01-05


In [5]:
# 1. Calendar features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_year"] = df["date"].dt.dayofyear
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["quarter"] = df["date"].dt.quarter
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["date"].dt.is_month_end.astype(int)

print("Calendar features added.")


Calendar features added.


In [6]:
# 2. Lag features
group_columns = ["store_nbr", "family"]

for lag in [1, 7, 14, 28]:
    df[f"sales_lag_{lag}"] = (
        df.groupby(group_columns)["sales"].shift(lag)
    )

display(
    df[
        [
            "store_nbr",
            "family",
            "date",
            "sales",
            "sales_lag_1",
            "sales_lag_7",
            "sales_lag_14",
            "sales_lag_28",
        ]
    ].head(35)
)


,store_nbr,family,date,sales,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28
0,1,AUTOMOTIVE,2013-01-01,0.00,NaN,NaN,NaN,NaN
1,1,AUTOMOTIVE,2013-01-02,2.00,0.00,NaN,NaN,NaN
2,1,AUTOMOTIVE,2013-01-03,3.00,2.00,NaN,NaN,NaN
3,1,AUTOMOTIVE,2013-01-04,3.00,3.00,NaN,NaN,NaN
4,1,AUTOMOTIVE,2013-01-05,5.00,3.00,NaN,NaN,NaN
5,1,AUTOMOTIVE,2013-01-06,2.00,5.00,NaN,NaN,NaN
6,1,AUTOMOTIVE,2013-01-07,0.00,2.00,NaN,NaN,NaN
7,1,AUTOMOTIVE,2013-01-08,2.00,0.00,0.00,NaN,NaN
8,1,AUTOMOTIVE,2013-01-09,2.00,2.00,2.00,NaN,NaN
9,1,AUTOMOTIVE,2013-01-10,2.00,2.00,3.00,NaN,NaN


In [7]:
# 3. Leakage-safe rolling features
# Shift sales first: rolling statistics only use previous observations.
previous_sales = (
    df.groupby(group_columns)["sales"].shift(1)
)

for window in [7, 14, 28]:
    grouped_previous_sales = previous_sales.groupby(
        [df["store_nbr"], df["family"]]
    )

    df[f"sales_rolling_mean_{window}"] = (
        grouped_previous_sales.transform(
            lambda x: x.rolling(window).mean()
        )
    )

    df[f"sales_rolling_std_{window}"] = (
        grouped_previous_sales.transform(
            lambda x: x.rolling(window).std()
        )
    )

print("Rolling features added.")


Rolling features added.


In [8]:
# 4. Store information
store_info = stores[
    ["store_nbr", "city", "state", "type", "cluster"]
].drop_duplicates("store_nbr")

df = df.merge(
    store_info,
    on="store_nbr",
    how="left",
    validate="many_to_one"
)

print("Store information merged.")


Store information merged.


In [9]:
# 5. Transactions
transaction_info = transactions[
    ["date", "store_nbr", "transactions"]
].drop_duplicates(["date", "store_nbr"])

df = df.merge(
    transaction_info,
    on=["date", "store_nbr"],
    how="left",
    validate="many_to_one"
)

df["transactions"] = df["transactions"].fillna(0)

print("Transactions merged.")


Transactions merged.


In [10]:
# 6. Oil price
oil_info = oil[
    ["date", "dcoilwtico"]
].sort_values("date")

df = pd.merge_asof(
    df.sort_values("date"),
    oil_info,
    on="date",
    direction="backward"
)

df["dcoilwtico"] = df["dcoilwtico"].ffill().bfill()

print("Oil prices merged.")


Oil prices merged.


In [11]:
# 7. Holiday information
holiday_summary = (
    holidays.groupby("date")
    .agg(
        holiday_count=("type", "size"),
        holiday_types=(
            "type",
            lambda x: "|".join(sorted(x.astype(str).unique()))
        )
    )
    .reset_index()
)

holiday_summary["is_holiday"] = (
    holiday_summary["holiday_count"] > 0
).astype(int)

df = df.merge(
    holiday_summary,
    on="date",
    how="left",
    validate="many_to_one"
)

df["holiday_count"] = df["holiday_count"].fillna(0)
df["is_holiday"] = df["is_holiday"].fillna(0)
df["holiday_types"] = df["holiday_types"].fillna("None")

print("Holiday information merged.")


Holiday information merged.


In [12]:
# Remove rows that do not yet have enough history for lag/rolling features.
history_columns = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_rolling_mean_7",
    "sales_rolling_std_7",
    "sales_rolling_mean_14",
    "sales_rolling_std_14",
    "sales_rolling_mean_28",
    "sales_rolling_std_28",
]

before = len(df)

df = df.dropna(
    subset=history_columns
).reset_index(drop=True)

after = len(df)

print("Rows before removing insufficient history:", before)
print("Rows after:", after)
print("Rows removed:", before - after)


Rows before removing insufficient history: 3000888
Rows after: 2950992
Rows removed: 49896


In [13]:
# Review the final feature list.
print("Final shape:", df.shape)
display(df.head())

print("\nMissing values in important features:")
display(
    df[history_columns].isna().sum().to_frame("missing_values")
)


Final shape: (2950992, 35)


,id,date,store_nbr,family,sales,onpromotion,year,month,day,day_of_week,day_of_year,week_of_year,quarter,is_weekend,is_month_start,is_month_end,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_std_7,sales_rolling_mean_14,sales_rolling_std_14,sales_rolling_mean_28,sales_rolling_std_28,city,state,type,cluster,transactions,dcoilwtico,holiday_count,holiday_types,is_holiday
0,51053,2013-01-29,41,BEAUTY,0.00,0,2013,1,29,1,29,5,1,0,0,0,4.00,1.00,3.00,0.00,2.00,1.29,2.07,1.27,1.89,1.42,Machala,El Oro,D,4,680.00,97.62,0.00,None,0.00
1,50800,2013-01-29,34,GROCERY II,3.00,0,2013,1,29,1,29,5,1,0,0,0,1.00,3.00,32.00,0.00,6.29,4.42,8.64,8.03,8.57,6.91,Guayaquil,Guayas,B,6,"2,306.00",97.62,0.00,None,0.00
2,50398,2013-01-29,23,CLEANING,470.00,0,2013,1,29,1,29,5,1,0,0,0,570.00,654.00,612.00,0.00,587.14,66.49,592.21,75.84,619.50,158.89,Ambato,Tungurahua,D,9,"1,139.00",97.62,0.00,None,0.00
3,50562,2013-01-29,28,CELEBRATION,0.00,0,2013,1,29,1,29,5,1,0,0,0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,Guayaquil,Guayas,E,10,815.00,97.62,0.00,None,0.00
4,49967,2013-01-29,11,BREAD/BAKERY,477.00,0,2013,1,29,1,29,5,1,0,0,0,519.00,440.00,462.00,0.00,513.71,56.13,514.14,54.08,512.82,138.10,Cayambe,Pichincha,B,6,"2,095.00",97.62,0.00,None,0.00



Missing values in important features:


,missing_values
sales_lag_1,0
sales_lag_7,0
sales_lag_14,0
sales_lag_28,0
sales_rolling_mean_7,0
sales_rolling_std_7,0
sales_rolling_mean_14,0
sales_rolling_std_14,0
sales_rolling_mean_28,0
sales_rolling_std_28,0


In [14]:
# Save the processed training dataset.
output_path = PROCESSED_DATA_DIR / "train_featured.csv"
df.to_csv(output_path, index=False)

print("Saved:", output_path)


Saved: d:\Projects\Revenue Demand Forecasting\data\processed\train_featured.csv


## Feature engineering complete

The processed dataset is now ready for modelling.

Saved file:

`data/processed/train_featured.csv`
